# 03 - 其他技术因子计算

**功能**: 计算动量、成交量、波动率、偏离度因子

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from scipy import stats

PROJECT_ROOT = Path('../../').resolve()
db_path = PROJECT_ROOT / 'data' / 'invest.db'
conn = sqlite3.connect(db_path)

def 计算分位数 (series, lookback=250):
    return series.rolling(lookback).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100 if len(x) > 10 else np.nan
    )

print("✅ 函数定义完成")

In [ ]:
# 读取数据
daily_df = pd.read_sql_query("SELECT * FROM stock_daily ORDER BY trade_date", conn)
daily_df['trade_date'] = pd.to_datetime(daily_df['trade_date'], format='%Y%m%d')

test_code = daily_df['ts_code'].iloc[0]
stock_data = daily_df[daily_df['ts_code'] == test_code].sort_values('trade_date').copy()
stock_data.set_index('trade_date', inplace=True)

close = stock_data['close']
volume = stock_data['vol']

print(f"测试股票：{test_code}")

In [ ]:
# 动量因子
stock_data['momentum_20'] = close / close.shift(20) - 1
stock_data['momentum_pct'] = 计算分位数 (stock_data['momentum_20'])
stock_data['momentum_accel'] = stock_data['momentum_20'] - stock_data['momentum_20'].shift(20)

# 成交量因子
volume_ma20 = volume.rolling(20).mean()
stock_data['volume_ratio'] = volume / volume_ma20
stock_data['volume_pct'] = 计算分位数 (volume)

# 波动率因子
returns = close.pct_change()
volatility = returns.rolling(20).std() * np.sqrt(252)
stock_data['volatility'] = volatility
stock_data['volatility_pct'] = 计算分位数 (volatility)

# 偏离度因子
ma20 = close.rolling(20).mean()
stock_data['deviation_20'] = (close - ma20) / ma20
stock_data['deviation_pct'] = 计算分位数 (stock_data['deviation_20'])

print("✅ 所有因子计算完成")

In [ ]:
# 可视化：因子相关性
import plotly.graph_objects as go

factor_cols = ['boll_position', 'macd_dif_pct', 'rsi_pct', 'momentum_pct', 'volume_pct', 'volatility_pct', 'deviation_pct']
corr_matrix = stock_data[factor_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=factor_cols,
    y=factor_cols,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate='%{text}'
))

fig.update_layout(title='技术因子相关性热力图', height=600)
fig.show()